In [ ]:
# ================================================================
# EXERCISE 10: ART NEURAL NETWORK
# Dataset: Dry Bean Dataset
# Algorithm: Fuzzy Adaptive Resonance Theory (Fuzzy ART)
# Run the complete program in ONE Google Colab cell.
# ================================================================

import sys
import subprocess
import importlib.util

# -------------------- INSTALL DEPENDENCIES -----------------------

for package, module in [
    ("ucimlrepo", "ucimlrepo"),
    ("openpyxl", "openpyxl")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import io
import zipfile
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)


# ================================================================
# 1. LOAD DRY BEAN DATASET
# ================================================================

def load_dataset():
    print("\nLoading Dry Bean Dataset...")

    # Method 1: UCI Machine Learning Repository API
    try:
        from ucimlrepo import fetch_ucirepo

        dataset = fetch_ucirepo(id=602)

        features = dataset.data.features
        targets = dataset.data.targets

        df = pd.concat([features, targets], axis=1)

        print("Dataset downloaded successfully from UCI.")
        return df

    except Exception as e:
        print("UCI download unavailable:", str(e)[:200])

    # Method 2: Upload CSV, Excel, or ZIP file
    try:
        from google.colab import files

        print("\nUpload Dry_Bean_Dataset.xlsx, CSV, or ZIP.")
        uploaded = files.upload()

        if not uploaded:
            raise RuntimeError("No file uploaded.")

        for filename, content in uploaded.items():
            lower = filename.lower()

            if lower.endswith(".csv"):
                df = pd.read_csv(io.BytesIO(content))

            elif lower.endswith((".xlsx", ".xls")):
                df = pd.read_excel(io.BytesIO(content))

            elif lower.endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(content)) as archive:
                    candidates = [
                        name for name in archive.namelist()
                        if name.lower().endswith(
                            (".csv", ".xlsx", ".xls")
                        )
                        and not name.startswith("__MACOSX/")
                    ]

                    if not candidates:
                        continue

                    selected = next(
                        (
                            name for name in candidates
                            if "dry_bean" in name.lower()
                        ),
                        candidates[0]
                    )

                    with archive.open(selected) as file:
                        if selected.lower().endswith(".csv"):
                            df = pd.read_csv(file)
                        else:
                            df = pd.read_excel(file)

            else:
                print("Unsupported file:", filename)
                continue

            print("Dataset loaded:", filename)
            return df

    except Exception as e:
        print("Upload/loading error:", str(e))

    raise RuntimeError(
        "Could not load the dataset. Please upload the Dry Bean dataset."
    )


df = load_dataset()

df.columns = [str(col).strip() for col in df.columns]
df = df.dropna().reset_index(drop=True)

# Identify class labels
class_col = next(
    (
        col for col in df.columns
        if col.strip().lower() in [
            "class", "bean_class", "bean class", "target"
        ]
    ),
    None
)

if class_col is None:
    # UCI Dry Bean data normally stores the class in the last column.
    if not all(
        pd.api.types.is_numeric_dtype(df[col])
        for col in df.columns
    ):
        class_col = df.columns[-1]

if class_col is not None:
    labels = df[class_col].astype(str).to_numpy()
    X_df = df.drop(columns=[class_col])
else:
    labels = None
    X_df = df.copy()

# Keep numeric input features
X_df = X_df.select_dtypes(include=[np.number])

if X_df.empty:
    raise ValueError("No numeric feature columns were found.")

X_df = X_df.replace([np.inf, -np.inf], np.nan)
valid_rows = X_df.notna().all(axis=1)

X_df = X_df.loc[valid_rows].reset_index(drop=True)

if labels is not None:
    labels = labels[valid_rows.to_numpy()]

# Scale features into the range [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_df).astype(float)

# Fuzzy ART complement coding: [x, 1-x]
X = np.concatenate([X_scaled, 1.0 - X_scaled], axis=1)

print("\nDataset loaded successfully!")
print("Number of samples:", len(X))
print("Original numeric features:", X_df.shape[1])
print("Complement-coded input dimensions:", X.shape[1])

if labels is not None:
    print("Bean varieties:", len(np.unique(labels)))
    print("Classes:", np.unique(labels))


# ================================================================
# 2. FUZZY ART NEURAL NETWORK
# ================================================================

class FuzzyART:

    def __init__(
        self,
        vigilance=0.75,
        choice=0.001,
        learning_rate=1.0
    ):
        self.vigilance = float(vigilance)
        self.choice = float(choice)
        self.learning_rate = float(learning_rate)

        self.weights = []
        self.assignments = []
        self.resonance_count = 0
        self.reset_count = 0
        self.new_category_count = 0
        self.match_history = []
        self.category_history = []

    @staticmethod
    def fuzzy_and(a, b):
        return np.minimum(a, b)

    def choice_function(self, sample, weight):
        numerator = np.sum(self.fuzzy_and(sample, weight))
        denominator = self.choice + np.sum(weight)

        return numerator / denominator

    def match_function(self, sample, weight):
        denominator = np.sum(sample)

        if denominator <= 1e-12:
            return 1.0

        numerator = np.sum(self.fuzzy_and(sample, weight))
        return numerator / denominator

    def fit_one(self, sample):
        sample = np.asarray(sample, dtype=float)

        if len(self.weights) == 0:
            self.weights.append(sample.copy())
            self.new_category_count += 1
            self.assignments.append(0)
            self.category_history.append(1)
            self.match_history.append(1.0)
            return 0

        candidates = [
            self.choice_function(sample, weight)
            for weight in self.weights
        ]

        order = np.argsort(candidates)[::-1]
        chosen_category = None

        for index in order:
            weight = self.weights[index]
            match = self.match_function(sample, weight)

            if match >= self.vigilance:
                self.resonance_count += 1
                self.match_history.append(match)

                # Update category prototype
                intersection = self.fuzzy_and(sample, weight)

                self.weights[index] = (
                    self.learning_rate * intersection
                    + (1.0 - self.learning_rate) * weight
                )

                chosen_category = int(index)
                break

            # Vigilance failure: reset and search another category
            self.reset_count += 1

        if chosen_category is None:
            chosen_category = len(self.weights)
            self.weights.append(sample.copy())
            self.new_category_count += 1
            self.match_history.append(1.0)

        self.assignments.append(chosen_category)
        self.category_history.append(len(self.weights))

        return chosen_category

    def fit(self, data):
        self.assignments = []
        self.category_history = []
        self.match_history = []
        self.resonance_count = 0
        self.reset_count = 0
        self.new_category_count = 0

        for sample in data:
            self.fit_one(sample)

        return np.array(self.assignments)

    def predict_one(self, sample):
        sample = np.asarray(sample, dtype=float)

        if len(self.weights) == 0:
            return None, 0.0

        candidates = [
            self.choice_function(sample, weight)
            for weight in self.weights
        ]

        for index in np.argsort(candidates)[::-1]:
            match = self.match_function(
                sample, self.weights[index]
            )

            if match >= self.vigilance:
                return int(index), float(match)

        return None, 0.0

    def predict(self, data):
        predictions = []
        matches = []

        for sample in data:
            category, match = self.predict_one(sample)
            predictions.append(
                -1 if category is None else category
            )
            matches.append(match)

        return np.array(predictions), np.array(matches)


# ================================================================
# 3. HELPER FUNCTIONS
# ================================================================

def train_art(data, vigilance=0.75, order=None):
    network = FuzzyART(
        vigilance=vigilance,
        choice=0.001,
        learning_rate=1.0
    )

    if order is None:
        order = np.arange(len(data))

    assignments = network.fit(data[order])
    return network, assignments


def print_network_summary(network, samples_seen):
    print("\n========== ART NETWORK SUMMARY ==========")
    print("Samples processed:", samples_seen)
    print("Categories formed:", len(network.weights))
    print("Vigilance:", network.vigilance)
    print("Resonance events:", network.resonance_count)
    print("Reset events:", network.reset_count)
    print("New categories:", network.new_category_count)
    print("=========================================")


def ask_vigilance(default=0.75):
    try:
        value = input(
            f"Enter vigilance between 0 and 1 [{default}]: "
        ).strip()

        value = default if value == "" else float(value)

        if not 0 <= value <= 1:
            raise ValueError

        return value

    except ValueError:
        print("Invalid value. Using", default)
        return default


def ask_sample_count(default=500):
    try:
        value = input(
            f"Number of samples to use [{default}]: "
        ).strip()

        value = default if value == "" else int(value)

        return max(1, min(len(X), value))

    except ValueError:
        print("Invalid value. Using", default)
        return min(default, len(X))


# ================================================================
# 4. MENU OPTION 1: INCREMENTAL CATEGORY FORMATION
# ================================================================

def incremental_category_formation():
    print("\n--- Incremental Bean Category Formation ---")

    n = ask_sample_count(500)
    vigilance = ask_vigilance(0.75)

    network = FuzzyART(vigilance=vigilance)

    for i in range(n):
        network.fit_one(X[i])

        if (i + 1) % max(1, n // 10) == 0:
            print(
                f"Processed {i+1}/{n} samples | "
                f"Categories: {len(network.weights)}"
            )

    print_network_summary(network, n)

    print("\nFirst 20 category assignments:")
    print(network.assignments[:20])


# ================================================================
# 5. MENU OPTION 2: VIGILANCE PARAMETER CHALLENGE
# ================================================================

def vigilance_parameter_challenge():
    print("\n--- Vigilance Parameter Challenge ---")

    n = min(len(X), 800)
    data = X[:n]

    vigilance_values = [0.30, 0.45, 0.60, 0.75, 0.90, 0.98]

    print(
        f"\n{'Vigilance':<12}"
        f"{'Categories':<15}"
        f"{'Resonance':<15}"
        f"{'Reset':<12}"
    )

    results = []

    for vigilance in vigilance_values:
        network, assignments = train_art(data, vigilance)

        print(
            f"{vigilance:<12.2f}"
            f"{len(network.weights):<15}"
            f"{network.resonance_count:<15}"
            f"{network.reset_count:<12}"
        )

        results.append(
            (vigilance, len(network.weights), network.reset_count)
        )

    print("\nInterpretation:")
    print("- Lower vigilance generally allows broader categories.")
    print("- Higher vigilance generally creates more specific categories.")
    print("- Category counts also depend on the data and input order.")

    if labels is not None:
        print("\nAdjusted Rand Index against known bean classes:")
        for vigilance in vigilance_values:
            network, assignments = train_art(data, vigilance)
            true_labels = labels[:n]
            score = adjusted_rand_score(true_labels, assignments)
            print(
                f"Vigilance {vigilance:.2f}: ARI = {score:.4f}"
            )


# ================================================================
# 6. MENU OPTION 3: RESONANCE AND RESET ANALYZER
# ================================================================

def resonance_reset_analyzer():
    print("\n--- Resonance and Reset Analyzer ---")

    n = ask_sample_count(500)
    vigilance = ask_vigilance(0.75)

    network, assignments = train_art(X[:n], vigilance)

    print_network_summary(network, n)

    total_events = (
        network.resonance_count + network.reset_count
    )

    if total_events > 0:
        print(
            "Resonance percentage:",
            round(
                100 * network.resonance_count / total_events, 2
            ),
            "%"
        )
        print(
            "Reset percentage:",
            round(
                100 * network.reset_count / total_events, 2
            ),
            "%"
        )

    print("\nCategory creation over time:")
    step = max(1, n // 10)

    for i in range(step - 1, n, step):
        print(
            f"Sample {i+1}: "
            f"{network.category_history[i]} categories"
        )

    if network.match_history:
        print(
            "\nAverage match score:",
            round(float(np.mean(network.match_history)), 4)
        )


# ================================================================
# 7. MENU OPTION 4: NOVEL BEAN PATTERN DETECTOR
# ================================================================

def novel_pattern_detector():
    print("\n--- Novel Bean Pattern Detector ---")

    n = min(len(X), 1000)
    split = max(1, int(n * 0.8))

    vigilance = ask_vigilance(0.80)

    network, _ = train_art(X[:split], vigilance)

    print("Categories learned:", len(network.weights))
    print("Testing previously unseen samples...")

    novel_count = 0
    known_count = 0

    for i in range(split, n):
        category, match = network.predict_one(X[i])

        if category is None:
            status = "NOVEL PATTERN"
            novel_count += 1

            # Learn the newly discovered category
            network.fit_one(X[i])

        else:
            status = "MATCHED CATEGORY"
            known_count += 1

        if i < split + 20:
            print(
                f"Sample {i+1}: {status} | "
                f"Category: {category} | "
                f"Match: {match:.3f}"
            )

    print("\nNovel patterns:", novel_count)
    print("Matched existing categories:", known_count)
    print("Categories after learning:", len(network.weights))


# ================================================================
# 8. MENU OPTION 5: CATEGORY SPLIT AND MERGE EXPLORER
# ================================================================

def category_split_merge_explorer():
    print("\n--- Category Split and Merge Explorer ---")

    n = min(len(X), 800)
    data = X[:n]

    low_vigilance = 0.45
    high_vigilance = 0.90

    low_network, low_assignments = train_art(
        data, low_vigilance
    )

    high_network, high_assignments = train_art(
        data, high_vigilance
    )

    print("\nLow vigilance:", low_vigilance)
    print("Categories formed:", len(low_network.weights))

    print("\nHigh vigilance:", high_vigilance)
    print("Categories formed:", len(high_network.weights))

    print(
        "\nAdditional categories at high vigilance:",
        len(high_network.weights) - len(low_network.weights)
    )

    print("\nCategory membership comparison:")

    for i in range(min(15, n)):
        print(
            f"Sample {i+1}: "
            f"Low vigilance category={low_assignments[i]}, "
            f"High vigilance category={high_assignments[i]}"
        )

    print(
        "\nNote: Fuzzy ART normally creates categories through "
        "vigilance-based resonance and reset. It does not perform "
        "automatic category merging in this implementation."
    )


# ================================================================
# 9. MENU OPTION 6: CATASTROPHIC FORGETTING RESISTANCE TEST
# ================================================================

def forgetting_resistance_test():
    print("\n--- Catastrophic Forgetting Resistance Test ---")

    n = min(len(X), 900)
    split = max(1, n // 3)

    A = X[:split]
    B = X[split:2 * split]
    C = X[2 * split:3 * split]

    vigilance = ask_vigilance(0.75)

    network = FuzzyART(vigilance=vigilance)

    network.fit(A)

    def retention(data):
        predictions, matches = network.predict(data)
        return float(np.mean(predictions != -1))

    initial_retention = retention(A)

    print("\nAfter learning group A:")
    print("Categories:", len(network.weights))
    print("A retention:", round(initial_retention * 100, 2), "%")

    network.fit(B)

    retention_after_B = retention(A)

    print("\nAfter learning group B:")
    print("Categories:", len(network.weights))
    print("A retention:", round(retention_after_B * 100, 2), "%")

    network.fit(C)

    retention_after_C = retention(A)

    print("\nAfter learning group C:")
    print("Categories:", len(network.weights))
    print("A retention:", round(retention_after_C * 100, 2), "%")

    print("\nRetention changes:")
    print(
        "After B:",
        round(
            (retention_after_B - initial_retention) * 100, 2
        ),
        "percentage points"
    )
    print(
        "After C:",
        round(
            (retention_after_C - initial_retention) * 100, 2
        ),
        "percentage points"
    )

    print(
        "\nRetention measures whether earlier samples still match "
        "some category at the selected vigilance."
    )


# ================================================================
# 10. MENU OPTION 7: INPUT ORDER SENSITIVITY CHALLENGE
# ================================================================

def input_order_sensitivity():
    print("\n--- Input Order Sensitivity Challenge ---")

    n = min(len(X), 700)
    data = X[:n]

    vigilance = ask_vigilance(0.75)

    order1 = np.arange(n)
    order2 = np.random.default_rng(42).permutation(n)
    order3 = np.random.default_rng(100).permutation(n)

    network1, assignments1 = train_art(
        data, vigilance, order1
    )

    network2, assignments2 = train_art(
        data, vigilance, order2
    )

    network3, assignments3 = train_art(
        data, vigilance, order3
    )

    print("\nResults:")

    print(
        "Sequential order categories:",
        len(network1.weights)
    )
    print(
        "Random order 1 categories:",
        len(network2.weights)
    )
    print(
        "Random order 2 categories:",
        len(network3.weights)
    )

    # Compare assignments after restoring original sample order
    restored2 = np.empty(n, dtype=int)
    restored2[order2] = assignments2

    restored3 = np.empty(n, dtype=int)
    restored3[order3] = assignments3

    print(
        "\nARI: sequential vs random order 1:",
        round(
            adjusted_rand_score(assignments1, restored2), 4
        )
    )

    print(
        "ARI: sequential vs random order 2:",
        round(
            adjusted_rand_score(assignments1, restored3), 4
        )
    )

    print(
        "\nDifferent input orders can produce different category "
        "prototypes and assignments."
    )


# ================================================================
# 11. MENU OPTION 8: ART CATEGORY FORMATION REPORT
# ================================================================

def category_formation_report():
    print("\n--- ART Category Formation Report ---")

    n = min(len(X), 1000)
    vigilance = ask_vigilance(0.75)

    network, assignments = train_art(X[:n], vigilance)

    print("\n========== FINAL REPORT ==========")
    print("Dataset: Dry Bean Dataset")
    print("Algorithm: Fuzzy ART")
    print("Samples processed:", n)
    print("Input features:", X_df.shape[1])
    print("Complement-coded dimensions:", X.shape[1])
    print("Vigilance:", vigilance)
    print("Categories formed:", len(network.weights))
    print("Resonance events:", network.resonance_count)
    print("Reset events:", network.reset_count)
    print("New categories:", network.new_category_count)

    if len(network.weights) > 0:
        sizes = np.bincount(
            assignments,
            minlength=len(network.weights)
        )

        print("Largest category size:", int(sizes.max()))
        print("Smallest category size:", int(sizes.min()))
        print("Average category size:", round(float(sizes.mean()), 2))

    if labels is not None:
        score = adjusted_rand_score(labels[:n], assignments)

        print(
            "Adjusted Rand Index against bean labels:",
            round(score, 4)
        )

        print(
            "Note: ARI compares discovered categories with known "
            "labels; the ART algorithm itself is unsupervised."
        )

    print("\nFirst 20 assignments:")
    print(assignments[:20])

    print("\nCategory sizes:")

    for category, size in enumerate(
        np.bincount(assignments, minlength=len(network.weights))
    ):
        print(f"Category {category}: {size} samples")

    print("==================================")


# ================================================================
# 12. INTERACTIVE MENU
# ================================================================

def main():
    while True:
        print("\n" + "=" * 55)
        print(" ART NEURAL NETWORK - DRY BEAN DATASET")
        print("=" * 55)
        print("1. Incremental Bean Category Formation")
        print("2. Vigilance Parameter Challenge")
        print("3. Resonance and Reset Analyzer")
        print("4. Novel Bean Pattern Detector")
        print("5. Category Split and Merge Explorer")
        print("6. Catastrophic Forgetting Resistance Test")
        print("7. Input Order Sensitivity Challenge")
        print("8. ART Category Formation Report")
        print("9. Exit")
        print("=" * 55)

        choice = input("Enter your choice (1-9): ").strip()

        try:
            if choice == "1":
                incremental_category_formation()

            elif choice == "2":
                vigilance_parameter_challenge()

            elif choice == "3":
                resonance_reset_analyzer()

            elif choice == "4":
                novel_pattern_detector()

            elif choice == "5":
                category_split_merge_explorer()

            elif choice == "6":
                forgetting_resistance_test()

            elif choice == "7":
                input_order_sensitivity()

            elif choice == "8":
                category_formation_report()

            elif choice == "9":
                print("Exiting ART Neural Network program. Goodbye!")
                break

            else:
                print("Invalid choice. Please enter a number from 1 to 9.")

        except Exception as e:
            print("\nAn error occurred:", str(e))
            print("Please try again.")

        if choice != "9":
            input("\nPress Enter to return to the main menu...")


main()


Loading Dry Bean Dataset...
Dataset downloaded successfully from UCI.

Dataset loaded successfully!
Number of samples: 13611
Original numeric features: 16
Complement-coded input dimensions: 32
Bean varieties: 7
Classes: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']

 ART NEURAL NETWORK - DRY BEAN DATASET
1. Incremental Bean Category Formation
2. Vigilance Parameter Challenge
3. Resonance and Reset Analyzer
4. Novel Bean Pattern Detector
5. Category Split and Merge Explorer
6. Catastrophic Forgetting Resistance Test
7. Input Order Sensitivity Challenge
8. ART Category Formation Report
9. Exit
Enter your choice (1-9): 9
Exiting ART Neural Network program. Goodbye!
